# B03 — Type 1 End-to-End Eval (via `/z3` API)

Sends each labelled instance to the EXACT API `/z3` endpoint and compares the
returned answer against the gold label.

- **Input:** `Logic_Based_Educational_Queries.json` — full labelled split with gold answers.
- **Pipeline:** parser vLLM → FOL ASTs → Z3 entailment → answer.
- **Scoring:** exact-match accuracy (overall + by question type), confusion matrix, latency.


In [1]:
import json, re, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

# --- endpoint -------------------------------------------------------------
API_BASE = "https://api.iamphuckhang.dev"   # VM via cloudflared tunnel
# API_BASE = "http://127.0.0.1:8080"         # only if kernel runs ON the VM
Z3_URL = f"{API_BASE}/z3"

# --- run size -------------------------------------------------------------
N_SAMPLES   = 20      # how many instances to eval (None = all 808)
CONCURRENCY = 8        # parallel in-flight requests
TIMEOUT     = 120.0    # per-request seconds

# --- locate dataset dir ---------------------------------------------------
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", Z3_URL)


dataset : /home/phuckhang/MyWorkspace/Exact2026/src/exact/datasets/exact
endpoint: https://api.iamphuckhang.dev/z3


In [2]:
raw = json.load(open(DATA / "Logic_Based_Educational_Queries.json"))

# Gold dataset uses "Unknown"; competition spec uses "Uncertain" — normalise.
GOLD_NORM = {"Unknown": "Uncertain"}

def parse_mcq_options(question_text: str) -> tuple[str, dict]:
    """Split 'Stem\nA. ...\nB. ...' into (stem, {A: ..., B: ...})."""
    lines = question_text.split("\n")
    stem_lines, options = [], {}
    for line in lines:
        m = re.match(r"^([A-D])\.\s*(.*)", line.strip())
        if m:
            options[m.group(1)] = m.group(2).strip()
        elif not options:
            stem_lines.append(line)
    return "\n".join(stem_lines).strip(), options

def flatten_instances(groups: list) -> list[dict]:
    """Expand each group's question list into individual instances."""
    instances = []
    for g_idx, group in enumerate(groups):
        for q_idx, (question, gold) in enumerate(zip(group["questions"], group["answers"])):
            stem, options = parse_mcq_options(question)
            instances.append({
                "id": f"logic_{g_idx:04d}_{q_idx:02d}",
                "premises": group["premises-NL"],
                "question": stem,
                "options": options or None,
                "gold": GOLD_NORM.get(gold, gold),
                "q_type": "mcq" if options else "ynu",
            })
    return instances

instances = flatten_instances(raw)
mcq_count = sum(1 for i in instances if i["q_type"] == "mcq")
ynu_count = sum(1 for i in instances if i["q_type"] == "ynu")
print(f"{len(instances)} instances  (MCQ: {mcq_count}, YNU: {ynu_count})")

ex = instances[0]
print(f"\nExample  : {ex['id']}  type={ex['q_type']}  gold={ex['gold']}")
print(f"Question : {ex['question']}")
if ex["options"]:
    for label, text in ex["options"].items():
        print(f"  {label}. {text}")
print(f"Premises : {len(ex['premises'])}")


808 instances  (MCQ: 346, YNU: 462)

Example  : logic_0000_00  type=mcq  gold=A
Question : Which conclusion follows with the fewest premises?
  A. If a Python project is not optimized, then it is not well-tested
  B. If all Python projects are optimized, then all Python projects are well-structured
  C. If a Python project is well-tested, then it must be clean and readable
  D. If a Python project is not optimized, then it does not follow PEP 8 standards
Premises : 14


In [3]:
async def call(client, sem, inst):
    payload = {
        "id": inst["id"],
        "query": inst["question"],
        "premises": inst["premises"],
    }
    if inst["options"]:
        payload["options"] = inst["options"]

    async with sem:
        t0 = time.perf_counter()
        err, pred, fol = None, None, None
        try:
            r = await client.post(Z3_URL, json=payload, timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
            pred = body.get("answer")
            fol  = body.get("fol")
        except Exception as e:
            err = repr(e)
        dt = time.perf_counter() - t0

    return {
        "id": inst["id"],
        "q_type": inst["q_type"],
        "gold": inst["gold"],
        "pred": pred,
        "correct": pred == inst["gold"],
        "fol": fol,
        "latency": dt,
        "error": err,
    }


async def run_eval(instances):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(inst):
            nonlocal done
            res = await call(client, sem, inst)
            done += 1
            if done % 10 == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(i) for i in instances))


In [5]:
subset = instances[:N_SAMPLES] if N_SAMPLES else instances
print(f"Evaluating {len(subset)} instances at concurrency {CONCURRENCY}...")
t0 = time.perf_counter()
results = await run_eval(subset)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["error"]]
success = [r for r in results if not r["error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")


Evaluating 20 instances at concurrency 8...
  20/20
Success : 20/20
Errors  : 0
Wall    : 67.7s


In [6]:
# --- Errors ---
if errors:
    print("=== Errors ===")
    for r in errors:
        print(f"  [{r['id']}] {r['error']}")
    print()

# --- Per-instance results ---
print("=== Per-instance results ===")
for r in success:
    mark = "✓" if r["correct"] else "✗"
    print(f"  {mark} {r['id']:20s}  type={r['q_type']:3s}  gold={r['gold']:10s}  pred={r['pred']}")

# --- FOL for first result ---
if success:
    print(f"\n=== FOL for {success[0]['id']} ===")
    for line in (success[0]["fol"] or "").split("\n"):
        print(" ", line)


=== Per-instance results ===
  ✗ logic_0000_00         type=mcq  gold=A           pred=Uncertain
  ✓ logic_0000_01         type=ynu  gold=Yes         pred=Yes
  ✓ logic_0001_00         type=mcq  gold=C           pred=C
  ✗ logic_0001_01         type=ynu  gold=Yes         pred=Uncertain
  ✓ logic_0002_00         type=mcq  gold=C           pred=C
  ✗ logic_0002_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0003_00         type=mcq  gold=A           pred=Uncertain
  ✗ logic_0003_01         type=ynu  gold=Yes         pred=Uncertain
  ✓ logic_0004_00         type=mcq  gold=C           pred=C
  ✗ logic_0004_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0005_00         type=mcq  gold=B           pred=Uncertain
  ✗ logic_0005_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0006_00         type=mcq  gold=C           pred=Uncertain
  ✗ logic_0006_01         type=ynu  gold=No          pred=Uncertain
  ✗ logic_0007_00         type=mcq  gold=B       

In [7]:
# --- Accuracy ---
def accuracy(rows):
    if not rows: return float("nan")
    return sum(r["correct"] for r in rows) / len(rows)

all_ok   = [r for r in success]
mcq_ok   = [r for r in success if r["q_type"] == "mcq"]
ynu_ok   = [r for r in success if r["q_type"] == "ynu"]

print("=== Accuracy ===")
print(f"  Overall : {accuracy(all_ok):.1%}  ({sum(r['correct'] for r in all_ok)}/{len(all_ok)})")
print(f"  MCQ     : {accuracy(mcq_ok):.1%}  ({sum(r['correct'] for r in mcq_ok)}/{len(mcq_ok)})")
print(f"  YNU     : {accuracy(ynu_ok):.1%}  ({sum(r['correct'] for r in ynu_ok)}/{len(ynu_ok)})")

# --- Prediction distribution ---
print("\n=== Prediction distribution ===")
pred_dist = Counter(r["pred"] for r in success)
for label, cnt in sorted(pred_dist.items(), key=lambda x: -x[1]):
    print(f"  {label:12s}: {cnt}")

# --- Confusion: gold → pred ---
print("\n=== Confusion (gold → pred) ===")
conf = Counter((r["gold"], r["pred"]) for r in success)
gold_labels = sorted({r["gold"] for r in success})
pred_labels = sorted({r["pred"] for r in success if r["pred"]})
header = f"{'gold\\pred':12s}" + "".join(f"{p:12s}" for p in pred_labels)
print(header)
for g in gold_labels:
    row = f"{g:12s}" + "".join(f"{conf.get((g,p),0):<12d}" for p in pred_labels)
    print(row)

# --- Latency ---
lat = [r["latency"] for r in success]
if lat:
    print(f"\n=== Latency ===")
    print(f"  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")


=== Accuracy ===
  Overall : 20.0%  (4/20)
  MCQ     : 30.0%  (3/10)
  YNU     : 10.0%  (1/10)

=== Prediction distribution ===
  Uncertain   : 16
  C           : 3
  Yes         : 1

=== Confusion (gold → pred) ===
gold\pred   C           Uncertain   Yes         
A           0           2           0           
B           0           4           0           
C           3           1           0           
No          0           2           0           
Yes         0           7           1           

=== Latency ===
  mean=19.94s  p50=17.71s  max=47.94s


In [4]:
# --- Re-run known-failing instances ---
KNOWN_FAILURES = {
    "logic_0008_01", "logic_0009_01", "logic_0013_00",
    "logic_0023_00", "logic_0023_01", "logic_0030_00",
    "logic_0030_01", "logic_0031_00", "logic_0032_00",
}

retry_subset = [i for i in instances if i["id"] in KNOWN_FAILURES]
print(f"Re-running {len(retry_subset)} previously-failing instances...")
t0 = time.perf_counter()
retry_results = await run_eval(retry_subset)
wall = time.perf_counter() - t0

retry_errors  = [r for r in retry_results if r["error"]]
retry_success = [r for r in retry_results if not r["error"]]
print(f"\nSuccess : {len(retry_success)}/{len(retry_results)}")
print(f"Errors  : {len(retry_errors)}")
print(f"Wall    : {wall:.1f}s")

if retry_errors:
    print("\n=== Still failing ===")
    for r in retry_errors:
        print(f"  [{r['id']}] {r['error']}")

print("\n=== Results ===")
for r in retry_success:
    mark = "✓" if r["correct"] else "✗"
    print(f"  {mark} {r['id']:20s}  type={r['q_type']:3s}  gold={r['gold']:10s}  pred={r['pred']}")

Re-running 9 previously-failing instances...
  9/9
Success : 9/9
Errors  : 0
Wall    : 52.8s

=== Results ===
  ✗ logic_0008_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0009_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0013_00         type=mcq  gold=C           pred=Uncertain
  ✗ logic_0023_00         type=mcq  gold=A           pred=Uncertain
  ✗ logic_0023_01         type=ynu  gold=No          pred=Uncertain
  ✓ logic_0030_00         type=mcq  gold=Uncertain   pred=Uncertain
  ✗ logic_0030_01         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0031_00         type=ynu  gold=Yes         pred=Uncertain
  ✗ logic_0032_00         type=ynu  gold=Yes         pred=Uncertain


In [ ]:
r = next(x for x in retry_results if x["id"] == "logic_0013_00")
print(r["pred"], "\n")
print(r["fol"])

## Notes
- Set `N_SAMPLES = None` to run all 808 instances.
- Gold `"Unknown"` is normalised to `"Uncertain"` to match the competition spec.
- MCQ options are parsed out of the embedded `A. / B. / C. / D.` lines in the question text.
- `fol` field in the response contains per-premise and per-option FOL strings for debugging.
- To inspect a single result: `next(r for r in results if r['id'] == 'logic_0000_00')`.
